In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
import numpy as np
import pandas as pd
import ydf
from sklearn.metrics import mean_squared_log_error

#1-Load Data
train_file_path = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv"
test_file_path = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv"

train_df = pd.read_csv(train_file_path)
test_df  = pd.read_csv(test_file_path)

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

#2-Prepare Target
label="SalePrice"
train_df[label]=pd.to_numeric(train_df[label], errors="coerce")

#Log transfrom target for better RMSLE performance
train_df[label]=np.log1p(train_df[label])

#3-Train/Validation Split
def split_dataset(dataset, test_ratio=0.20, seed=42):
    np.random.seed(seed)
    test_indices=np.random.rand(len(dataset)) < test_ratio
    return dataset[~test_indices].copy(), dataset[test_indices].copy()

train_ds_pd, valid_ds_pd = split_dataset(train_df,test_ratio=0.2, seed=42)

print(f"{len(train_ds_pd)} examples in training")
print(f"{len(valid_ds_pd)} examples in validation")

#4-Train YDF Regression Model
model=ydf.GradientBoostedTreesLearner(
    label=label,
    task=ydf.Task.REGRESSION,
    num_trees=300,
    max_depth=6,
    min_examples=5,
    shrinkage=0.05,
    random_seed=42,
).train(train_ds_pd)

#5-Evaluate with YDF
evaluation=model.evaluate(valid_ds_pd)
print("\nYDF Evaluation:")
print(evaluation)

#6-Evaluate with Kaggle metric idea(RMSLE)
# since target is log1p(SalePrice), RMSE in log-space =RMSLE
#We will also compute RMSLE back on original scale

valid_pred_log = model.predict(valid_ds_pd)

valid_true=np.expm1(valid_ds_pd[label].values)
valid_pred=np.expm1(valid_pred_log)#avoid negative predictions before RMSLE
valid_pred=np.maximum(valid_pred,0)

rmsle=np.sqrt(mean_squared_log_error(valid_true,valid_pred))
print("\nValidation RMSLE:",rmsle)

#7-Feature Importance
print("\Feature Importances:")
try:
    print(model.variable_importances())
except Exception as e:
    print("Could not print feature importances:",e)

#Predict on test set
test_pred_log=model.predict(test_df)
test_pred=np.expm1(test_pred_log)
test_pred=np.maximum(test_pred, 0)

#Create Submission
submission=pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice":test_pred
})

submission.to_csv("submission.csv", index=False)
print("\Submission preview:")
print(submission.head())
print("\n submission.csv created successfully.")



<>:64: SyntaxWarning: invalid escape sequence '\F'
<>:82: SyntaxWarning: invalid escape sequence '\S'
<>:64: SyntaxWarning: invalid escape sequence '\F'
<>:82: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_17/1184836653.py:64: SyntaxWarning: invalid escape sequence '\F'
  print("\Feature Importances:")
/tmp/ipykernel_17/1184836653.py:82: SyntaxWarning: invalid escape sequence '\S'
  print("\Submission preview:")


Train Shape: (1460, 81)
Test Shape: (1459, 80)
1142 examples in training
318 examples in validation
Feature Street is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Feature Utilities is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Feature Condition2 is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Feature PoolQC is a CATEGORICAL feature with an empty dictionary. The feature will not be useful during model training.
Feature MiscFeature is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Train model on 1142 examples
Model trained in 0:00:04.445375

YDF Evaluation:
RMSE: 0.115444
num examples: 318
num examples (weighted): 318


Validation RMSLE: 0.11544443832997621
\Feature Importances:
{'NUM_NODES': [(648.0, 'Neighborho